# Berlin Döner-Standortanalyse — Master-Datenbasis

**Notebook 3 der Datenbeschaffungs-Pipeline** — Master Merge

Dieses Notebook konsolidiert alle Datenquellen auf PLR-Ebene (542 Planungsräume)
und erzeugt den finalen Analysedatensatz für das BI-Dashboard.

**Outputs:**
- `berlin_masterdata.csv` — Flache CSV-Tabelle (542 Zeilen, ~35 Spalten)
- `berlin_masterdata.db` — SQLite-Datenbank (Tabelle `planungsraeume`)

**Datenquellen und Join-Strategie:**

| Schritt | Quelle | Join-Schlüssel | Neue Spalten |
|---------|--------|----------------|-------------|
| 1 | LOR GeoJSON (`lor_planungsraeume_2021.geojson`) | Basis | plr_id, plr_name, bezirk, flaeche_km2, centroid |
| 2 | Bevölkerung CSV (`lor_bevoelkerungs-daten_2024.csv`) | RAUMID = PLR_ID | einwohner_gesamt, einwohner_18_35, einwohner_dichte |
| 3 | Medianeinkommen Excel (`lor_Medianeinkommen_31-12-2023.xlsx`) | RAUMID = PLR_ID | medianeinkommen_eur |
| 4 | MSS 2023 Excel (`lor_monitoring_soziale-stadtentwicklung_2023.xlsx`) | Nummer = PLR_ID | mss_status_index, mss_dynamik_label |
| 5 | Dönerläden CSV + JSON (`dataset_berlin_doener_clean.csv`) | Point-in-Polygon | doener_count, doener_avg_rating, Öffnungszeiten, Preis |
| 6 | IHK Gewerbedaten CSV (`lor_IHKBerlin_Gewerbedaten.csv`) | planungsraum_id = PLR_ID | gastro_gesamt, gastro_neu |
| 7 | Infrastruktur CSV (`lor_infrastruktur.csv`) | plr_id = PLR_ID | transit_count, university_count, school_count, ... |

**Voraussetzung:** Alle Input-Dateien müssen in `_file_name_config.txt` korrekt konfiguriert sein.

In [29]:
from pathlib import Path

# Dateinamen aus der Konfigurationsdatei laden
# Alle Pfade relativ zum Data_Pipeline-Ordner (siehe _file_name_config.txt)
CONFIG_FILE = Path('_file_name_config.txt')
config = {}
with open(CONFIG_FILE, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#'):  # Kommentarzeilen und Leerzeilen ueberspringen
            key, _, val = line.partition('=')
            config[key.strip()] = val.strip()

BASE = CONFIG_FILE.resolve().parent
LOR_GEOJSON       = BASE / config['LOR_GEOJSON']
POPULATION_CSV    = BASE / config['POPULATION_CSV']
INCOME_XLSX       = BASE / config['INCOME_XLSX']
MSS_XLSX          = BASE / config['MSS_XLSX']
DOENER_CSV        = BASE / config['DOENER_CSV']
IHK_CSV           = BASE / config['IHK_CSV']
INFRASTRUKTUR_CSV = BASE / config['INFRASTRUKTUR_CSV']
OUTPUT_CSV        = BASE / config['OUTPUT_CSV']
OUTPUT_SQLITE     = BASE / config['OUTPUT_SQLITE']

print('Konfiguration geladen:')
for k, v in config.items():
    p = BASE / v
    status = 'OK   ' if p.exists() else 'FEHLT'
    print('  [' + status + '] ' + k + ' = ' + v)


Konfiguration geladen:
  [OK   ] LOR_GEOJSON = lor_planungsraeume_2021.geojson
  [OK   ] POPULATION_CSV = lor_bevoelkerungs-daten_2024.csv
  [OK   ] INCOME_XLSX = lor_Medianeinkommen_31-12-2023.xlsx
  [OK   ] MSS_XLSX = lor_monitoring_soziale-stadtentwicklung_2023.xlsx
  [OK   ] DOENER_CSV = dataset_berlin_doener_clean.csv
  [OK   ] IHK_CSV = lor_IHKBerlin_Gewerbedaten.csv
  [OK   ] INFRASTRUKTUR_CSV = lor_infrastruktur.csv
  [OK   ] OUTPUT_CSV = berlin_masterdata.csv
  [OK   ] OUTPUT_SQLITE = berlin_masterdata.db


In [30]:
import json, csv, math, sqlite3, statistics
import pandas as pd
from datetime import datetime

print('Imports OK')


Imports OK


In [31]:
# ============================================================
# Hilfsfunktionen: Koordinaten-Konvertierung + Point-in-Polygon
# ============================================================

def utm33n_to_wgs84(easting, northing):
    # EPSG:25833 -> WGS84
    a = 6378137.0; f = 1/298.257223563
    e2 = 2*f - f**2; e_p2 = e2/(1-e2)
    k0 = 0.9996; E0 = 500000.0; lam0 = math.radians(15.0)
    E = easting - E0; N = northing
    mu = (N/k0) / (a*(1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))
    e1 = (1 - math.sqrt(1-e2)) / (1 + math.sqrt(1-e2))
    p1 = (mu + (3*e1/2 - 27*e1**3/32)*math.sin(2*mu)
              + (21*e1**2/16 - 55*e1**4/32)*math.sin(4*mu)
              + (151*e1**3/96)*math.sin(6*mu)
              + (1097*e1**4/512)*math.sin(8*mu))
    N1 = a/math.sqrt(1 - e2*math.sin(p1)**2)
    T1 = math.tan(p1)**2; C1 = e_p2*math.cos(p1)**2
    R1 = a*(1-e2)/(1 - e2*math.sin(p1)**2)**1.5
    D  = E/(N1*k0)
    lat = p1 - (N1*math.tan(p1)/R1)*(D**2/2
          - (5+3*T1+10*C1-4*C1**2-9*e_p2)*D**4/24
          + (61+90*T1+298*C1+45*T1**2-252*e_p2-3*C1**2)*D**6/720)
    lng = lam0 + (D - (1+2*T1+C1)*D**3/6
          + (5-2*C1+28*T1-3*C1**2+8*e_p2+24*T1**2)*D**5/120)/math.cos(p1)
    return math.degrees(lat), math.degrees(lng)

def wgs84_to_utm33n(lat_deg, lng_deg):
    # WGS84 -> EPSG:25833
    a = 6378137.0; f = 1/298.257223563
    e2 = 2*f - f**2; k0 = 0.9996; E0 = 500000.0; lam0 = math.radians(15.0)
    lat = math.radians(lat_deg); lng = math.radians(lng_deg)
    N = a/math.sqrt(1 - e2*math.sin(lat)**2)
    T = math.tan(lat)**2; C = (e2/(1-e2))*math.cos(lat)**2
    A = (lng - lam0)*math.cos(lat)
    M = a*((1 - e2/4 - 3*e2**2/64 - 5*e2**3/256)*lat
           - (3*e2/8 + 3*e2**2/32 + 45*e2**3/1024)*math.sin(2*lat)
           + (15*e2**2/256 + 45*e2**3/1024)*math.sin(4*lat)
           - (35*e2**3/3072)*math.sin(6*lat))
    easting  = k0*N*(A + (1-T+C)*A**3/6
               + (5-18*T+T**2+72*C-58*(e2/(1-e2)))*A**5/120) + E0
    northing = k0*(M + N*math.tan(lat)*(A**2/2
               + (5-T+9*C+4*C**2)*A**4/24
               + (61-58*T+T**2+600*C-330*(e2/(1-e2)))*A**6/720))
    return easting, northing

def point_in_polygon(px, py, ring):
    # Ray-casting Algorithmus
    inside = False
    n = len(ring); j = n - 1
    for i in range(n):
        xi, yi = ring[i][0], ring[i][1]
        xj, yj = ring[j][0], ring[j][1]
        if ((yi > py) != (yj > py)) and (px < (xj-xi)*(py-yi)/(yj-yi) + xi):
            inside = not inside
        j = i
    return inside

BEZIRK_NAMEN = {
    '01': 'Mitte', '02': 'Friedrichshain-Kreuzberg', '03': 'Pankow',
    '04': 'Charlottenburg-Wilmersdorf', '05': 'Spandau',
    '06': 'Steglitz-Zehlendorf', '07': 'Tempelhof-Schoeneberg',
    '08': 'Neukoelln', '09': 'Treptow-Koepenick',
    '10': 'Marzahn-Hellersdorf', '11': 'Lichtenberg', '12': 'Reinickendorf',
}

print('Hilfsfunktionen OK')


Hilfsfunktionen OK


## Schritt 1 — LOR Basis: Geometrie + Zentroide + Bezirksnamen

Lädt die LOR-Planungsräume aus dem GeoJSON (EPSG:25833, UTM Zone 33N).
Für jeden der 542 PLR werden berechnet:
- **Zentroid** (Durchschnitt aller Polygon-Vertices, dann UTM→WGS84)
- **Fläche** aus dem `GROESSE_M2`-Attribut
- **Bezirksname** über eine Lookup-Tabelle (BEZ-Nummer -> Name)

Die Ring-Geometrie (`plr_rings`) wird für den Point-in-Polygon-Check in Schritt 5 beibehalten.

In [32]:
# ============================================================
# Schritt 1 — LOR Basis: Geometrie + Zentroide + Bezirksnamen
# ============================================================

with open(LOR_GEOJSON, encoding='utf-8') as f:
    geojson = json.load(f)

plr_rows  = {}   # Haupttabelle: plr_id -> dict
plr_rings = {}   # Geometrie für Point-in-Polygon (Schritt 5)

for feat in geojson['features']:
    props  = feat['properties']
    pid    = props['PLR_ID']
    ring   = feat['geometry']['coordinates'][0][0]
    xs = [c[0] for c in ring]; ys = [c[1] for c in ring]
    clat, clng = utm33n_to_wgs84(sum(xs)/len(xs), sum(ys)/len(ys))
    bez_nr = str(props['BEZ']).zfill(2)
    plr_rows[pid] = {
        'plr_id':       pid,
        'plr_name':     props['PLR_NAME'],
        'bezirk':       BEZIRK_NAMEN.get(bez_nr, 'Unbekannt'),
        'flaeche_km2':  round(props['GROESSE_M2'] / 1_000_000, 4),
        'centroid_lat': round(clat, 6),
        'centroid_lng': round(clng, 6),
    }
    plr_rings[pid] = {
        'ring': ring,
        'bbox': (min(xs), min(ys), max(xs), max(ys)),
    }

print('PLR geladen: ' + str(len(plr_rows)))


PLR geladen: 542


## Schritt 2 — Bevölkerung (EWR_L21, Stand 31.12.2024)

Quelle: Amt für Statistik Berlin-Brandenburg, Einwohnerregisterstatistik.

Join: `RAUMID` (CSV) = `PLR_ID` (GeoJSON)

Neue Spalten:
- `einwohner_gesamt` — Gesamtbevölkerung im PLR (`E_E`-Spalte)
- `einwohner_18_35` — Summe der Altersgruppen 18-21, 21-25, 25-27, 27-30, 30-35
  (Proxy für die Döner-Kernzielgruppe)
- `einwohner_dichte` — Einwohner pro km² (einwohner_gesamt / flaeche_km2)

In [33]:
# ============================================================
# Schritt 2 — Bevölkerung (EWR_L21, Stand 31.12.2024)
# ============================================================

df_pop = pd.read_csv(POPULATION_CSV, sep=';', dtype={'RAUMID': str})
AGE_18_35 = ['E_E18_21', 'E_E21_25', 'E_E25_27', 'E_E27_30', 'E_E30_35']

joined = 0
for _, row in df_pop.iterrows():
    pid = str(row['RAUMID']).strip()
    if pid not in plr_rows:
        continue
    ew      = row.get('E_E')
    age_sum = sum(row[c] for c in AGE_18_35 if c in row.index and pd.notna(row[c]))
    flaeche = plr_rows[pid]['flaeche_km2']
    plr_rows[pid]['einwohner_gesamt']  = int(ew)    if pd.notna(ew)    else None
    plr_rows[pid]['einwohner_18_35']   = int(age_sum) if pd.notna(age_sum) else None
    plr_rows[pid]['einwohner_dichte']  = round(ew / flaeche, 1) if pd.notna(ew) and flaeche else None
    joined += 1

print('Bevölkerung gejoint: ' + str(joined) + '/' + str(len(plr_rows)) + ' PLR')


Bevölkerung gejoint: 541/542 PLR


## Schritt 3 — Medianeinkommen (Stand 31.12.2023)

Quelle: Senatsverwaltung für Finanzen, Lohnsteuerstatistik.

Join: `RAUMID` (Excel) = `PLR_ID` (Basis)

Neue Spalten:
- `medianeinkommen_eur` — Medianes Jahreseinkommen in EUR im PLR

**Hinweis:** Die Excel-Datei liegt eine Ebene höher (`../lor_Medianeinkommen_31-12-2023.xlsx`).
Zeilen 1-6 werden übersprungen (Header-Block), ab Zeile 7 stehen die PLR-Daten.

In [34]:
# ============================================================
# Schritt 3 — Medianeinkommen (Stand 31.12.2023)
# Quelle: lor_Medianeinkommen_31-12-2023.xlsx
# Join-Key: RAUMID = PLR_ID
# Neue Spalte: medianeinkommen_eur
# ============================================================

df_inc = pd.read_excel(INCOME_XLSX, sheet_name='Medianeinkommen PLR',
                       skiprows=6, dtype={'RAUMID': str})
df_inc = df_inc[['RAUMID', 'Medianeinkommen in EUR']].dropna(subset=['RAUMID'])
df_inc['RAUMID'] = df_inc['RAUMID'].str.strip()

joined = 0
for _, row in df_inc.iterrows():
    pid = str(row['RAUMID']).strip()
    if pid not in plr_rows:
        continue
    val = row['Medianeinkommen in EUR']
    # Excel verwendet '•' als Platzhalter fuer fehlende Werte -> als None behandeln
    try:
        plr_rows[pid]['medianeinkommen_eur'] = int(val) if pd.notna(val) else None
    except (ValueError, TypeError):
        plr_rows[pid]['medianeinkommen_eur'] = None
    joined += 1

print('Medianeinkommen gejoint: ' + str(joined) + '/' + str(len(plr_rows)) + ' PLR')


Medianeinkommen gejoint: 540/542 PLR


## Schritt 4 — Monitoring Soziale Stadtentwicklung 2023 (MSS)

Quelle: Senatsverwaltung für Stadtentwicklung, MSS 2023.

Join: Spalte 0 (PLR-Nummer im Excel) = `PLR_ID` (Basis)

Neue Spalten:
- `mss_status_index` — Sozialer Statusindex: 1=hoch, 2=mittel, 3=niedrig, 4=sehr niedrig
- `mss_dynamik_label` — Veränderungsdynamik: positiv, stabil, negativ

**Hinweis:** Die Tabelle hat 5 Überschriftzeilen (`skiprows=5`) und keinen Spalten-Header,
daher `header=None`. Spalte 0 = PLR-Nummer, Spalte 3 = Statusindex, Spalte 6 = Dynamiklabel.

In [35]:
# ============================================================
# Schritt 4 — Monitoring Soziale Stadtentwicklung 2023
# Status-Index: 1=hoch, 2=mittel, 3=niedrig, 4=sehr niedrig
# Dynamik-Label: positiv, stabil, negativ
# ============================================================

df_mss = pd.read_excel(MSS_XLSX, sheet_name='1.SDI_PLR_MSS2023',
                       skiprows=5, header=None, dtype={0: str})

joined = 0
for _, row in df_mss.iterrows():
    pid = str(row[0]).strip() if pd.notna(row[0]) else ''
    if pid not in plr_rows:
        continue
    plr_rows[pid]['mss_status_index']  = int(row[3]) if pd.notna(row[3]) else None
    plr_rows[pid]['mss_dynamik_label'] = str(row[6]).strip() if pd.notna(row[6]) else None
    joined += 1

print('MSS gejoint: ' + str(joined) + '/' + str(len(plr_rows)) + ' PLR')


MSS gejoint: 542/542 PLR


## Schritt 5 — Dönerläden Point-in-Polygon + Öffnungszeiten-Analyse

Quelle: `dataset_berlin_doener_clean.csv` (aus Notebook 1)

**Point-in-Polygon-Methode:**
1. Koordinaten jedes Dönerladens (WGS84) werden nach EPSG:25833 (UTM 33N) konvertiert
2. Bounding-Box-Prüfung (schnelle Vorfilterung)
3. Ray-Casting-Algorithmus für exakte Polygon-Zuordnung

**Aggregierte Basis-Spalten:**
- `doener_count` — Anzahl Dönerläden im PLR
- `doener_avg_rating` — Durchschnittliches Google-Rating
- `doener_total_reviews` — Summe aller Bewertungen
- `doener_best_rating` — Höchstes Rating im PLR
- `doener_pct_delivery` — Anteil Shops mit Lieferservice (%)

**Zusätzliche Öffnungszeiten- und Preis-Spalten** (aus `regularOpeningHours_periods` JSON):
- `doener_avg_hours_per_week` — Durchschnittliche Wochenstunden pro Shop im PLR
- `doener_pct_late_night` — Anteil Shops die nach 22:00 Uhr offen sind (%)
- `doener_pct_open_sunday` — Anteil Shops die sonntags offen sind (day=0)
- `doener_avg_price_level` — Durchschnittlicher Preislevel (priceRange_start_units)

In [36]:
# ============================================================
# Schritt 5 — Dönerläden Point-in-Polygon
# WGS84-Koordinaten -> EPSG:25833 -> Polygon-Check
# ============================================================

df_doener = pd.read_csv(DOENER_CSV, dtype=str, low_memory=False)
cols_lower = {c.lower(): c for c in df_doener.columns}
lat_col     = cols_lower.get('latitude')   or cols_lower.get('lat')
lng_col     = cols_lower.get('longitude')  or cols_lower.get('lng')
rating_col  = cols_lower.get('rating')
reviews_col = cols_lower.get('userratingcount') or cols_lower.get('user_rating_count')
delivery_col= cols_lower.get('delivery')
hours_col   = cols_lower.get('regularopeninghours_periods')  # JSON-String mit Öffnungszeiten
price_col   = cols_lower.get('pricerange_start_units')       # numerischer Preislevel

print('Spalten: lat=' + str(lat_col) + '  lng=' + str(lng_col))
print('Starte Point-in-Polygon für ' + str(len(df_doener)) + ' Läden...')

plr_doener  = {pid: [] for pid in plr_rows}
not_matched = 0

for idx, shop in df_doener.iterrows():
    try:
        lat = float(shop[lat_col]); lng = float(shop[lng_col])
    except (ValueError, TypeError):
        continue
    ex, ny  = wgs84_to_utm33n(lat, lng)  # WGS84 -> UTM 33N fuer Polygon-Check
    matched = None
    for pid, geo in plr_rings.items():
        bx0, by0, bx1, by1 = geo['bbox']
        if ex < bx0 or ex > bx1 or ny < by0 or ny > by1:  # Bounding-Box-Vorfilter
            continue
        if point_in_polygon(ex, ny, geo['ring']):  # Ray-Casting-Check
            matched = pid; break
    if matched:
        try:    r  = float(shop[rating_col])  if rating_col  and pd.notna(shop[rating_col])  else None
        except: r  = None
        try:    rv = int(float(shop[reviews_col])) if reviews_col and pd.notna(shop[reviews_col]) else 0
        except: rv = 0
        try:    dl = str(shop[delivery_col]).lower() == 'true' if delivery_col else False
        except: dl = False
        # Öffnungszeiten-Parsing: JSON-String -> Liste von Perioden
        hours_json = shop.get(hours_col) if hours_col else None
        periods = []
        if hours_json and pd.notna(hours_json):
            try:
                periods = json.loads(hours_json)
            except (json.JSONDecodeError, TypeError):
                periods = []
        # Preislevel direkt aus Spalte lesen (numerisch, z.B. 1-4)
        price_val = None
        if price_col:
            try:
                pv = shop.get(price_col)
                price_val = float(pv) if pv is not None and pd.notna(pv) else None
            except (ValueError, TypeError):
                price_val = None
        plr_doener[matched].append({
            'rating': r, 'reviews': rv, 'delivery': dl,
            'periods': periods, 'price_level': price_val
        })
    else:
        not_matched += 1

total_assigned = sum(len(v) for v in plr_doener.values())
print('Zugeordnet: ' + str(total_assigned) + '  Nicht zugeordnet: ' + str(not_matched))


def parse_hours_per_week(periods):
    """Berechnet die Gesamtstunden pro Woche aus den Öffnungszeiten-Perioden.
    Jede Periode hat open.day/hour/minute und close.day/hour/minute.
    Bei Mitternachts-Übergang (close.day != open.day) werden 24h addiert."""
    total = 0.0
    for p in periods:
        o = p.get('open', {})
        c = p.get('close', {})
        open_h  = o.get('hour', 0) + o.get('minute', 0) / 60.0
        close_h = c.get('hour', 0) + c.get('minute', 0) / 60.0
        if c.get('day', o.get('day', 0)) != o.get('day', 0):  # Mitternachts-Wrap
            close_h += 24
        total += max(close_h - open_h, 0)
    return total


def is_late_night(periods):
    """True wenn mindestens eine Periode nach 22:00 Uhr schliesst
    oder bis nach Mitternacht offen ist."""
    for p in periods:
        c = p.get('close', {})
        o = p.get('open', {})
        if c.get('hour', 0) >= 22:  # Schliesszeit >= 22 Uhr
            return True
        if c.get('day', o.get('day', 0)) != o.get('day', 0):  # über Mitternacht
            return True
    return False


def is_open_sunday(periods):
    """True wenn mindestens eine Periode am Sonntag beginnt.
    Google Places: day=0 ist Sonntag, day=1 ist Montag, etc."""
    for p in periods:
        if p.get('open', {}).get('day') == 0:
            return True
    return False


# Aggregation pro PLR
for pid, shops in plr_doener.items():
    if not shops:
        plr_rows[pid].update({
            'doener_count': 0,
            'doener_avg_rating': None,
            'doener_total_reviews': 0,
            'doener_best_rating': None,
            'doener_pct_delivery': None,
            'doener_avg_hours_per_week': None,
            'doener_pct_late_night': None,
            'doener_pct_open_sunday': None,
            'doener_avg_price_level': None,
        })
        continue

    ratings   = [s['rating'] for s in shops if s['rating'] is not None]
    # Öffnungszeiten nur fuer Shops mit validen Perioden berechnen
    shops_w_hours = [s for s in shops if s['periods']]
    hours_vals    = [parse_hours_per_week(s['periods']) for s in shops_w_hours]
    late_vals     = [is_late_night(s['periods']) for s in shops_w_hours]
    sun_vals      = [is_open_sunday(s['periods']) for s in shops_w_hours]
    price_vals    = [s['price_level'] for s in shops if s['price_level'] is not None]

    plr_rows[pid]['doener_count']             = len(shops)
    plr_rows[pid]['doener_avg_rating']        = round(sum(ratings)/len(ratings), 2) if ratings else None
    plr_rows[pid]['doener_total_reviews']     = sum(s['reviews'] for s in shops)
    plr_rows[pid]['doener_best_rating']       = max(ratings) if ratings else None
    plr_rows[pid]['doener_pct_delivery']      = round(sum(1 for s in shops if s['delivery'])/len(shops)*100, 1)
    plr_rows[pid]['doener_avg_hours_per_week']= round(sum(hours_vals)/len(hours_vals), 1) if hours_vals else None
    plr_rows[pid]['doener_pct_late_night']    = round(sum(late_vals)/len(late_vals)*100, 1) if late_vals else None
    plr_rows[pid]['doener_pct_open_sunday']   = round(sum(sun_vals)/len(sun_vals)*100, 1) if sun_vals else None
    plr_rows[pid]['doener_avg_price_level']   = round(sum(price_vals)/len(price_vals), 2) if price_vals else None

print('Aggregation abgeschlossen.')
doener_plr = sum(1 for pid in plr_rows if plr_rows[pid].get('doener_count', 0) > 0)
print('PLR mit mind. 1 Dönerladen: ' + str(doener_plr) + '/' + str(len(plr_rows)))


Spalten: lat=latitude  lng=longitude
Starte Point-in-Polygon für 1346 Läden...
Zugeordnet: 1229  Nicht zugeordnet: 117
Aggregation abgeschlossen.
PLR mit mind. 1 Dönerladen: 400/542


## Schritt 6 — IHK Gewerbedaten (Gastronomie NACE 56*)

Quelle: IHK Berlin, Gewerbemeldedaten.

Join: `planungsraum_id` (CSV, nullgepadded auf 8 Stellen) = `PLR_ID` (Basis)

Neue Spalten:
- `gastro_gesamt` — Anzahl aktiver Gastronomiebetriebe im PLR (NACE 56*)
- `gastro_neu` — Davon jünger als 2 Jahre (Proxy für Gastronomie-Dynamik)

In [37]:
# ============================================================
# Schritt 6 — IHK Gewerbedaten (Gastronomie NACE 56*)
# gastro_gesamt: alle aktiven Gastro-Betriebe im PLR
# gastro_neu:    business_age <= 2 Jahre (Dynamik-Proxy)
# ============================================================

plr_gastro = {pid: {'gesamt': 0, 'neu': 0} for pid in plr_rows}

with open(IHK_CSV, encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if not str(row.get('nace_id', '')).startswith('56'):
            continue
        pid = str(row.get('planungsraum_id', '')).strip().zfill(8)
        if pid not in plr_gastro:
            continue
        plr_gastro[pid]['gesamt'] += 1
        try:
            if int(row.get('business_age', 99)) <= 2:
                plr_gastro[pid]['neu'] += 1
        except (ValueError, TypeError):
            pass

for pid, vals in plr_gastro.items():
    plr_rows[pid]['gastro_gesamt'] = vals['gesamt']
    plr_rows[pid]['gastro_neu']    = vals['neu']

total = sum(v['gesamt'] for v in plr_gastro.values())
neu   = sum(v['neu']    for v in plr_gastro.values())
print('Gastro gesamt: ' + str(total) + '  davon neu (<=2 Jahre): ' + str(neu))


Gastro gesamt: 20372  davon neu (<=2 Jahre): 7238


## Schritt 7 — Infrastruktur (Google Places Aggregate API)

Quelle: `lor_infrastruktur.csv` (erzeugt durch Notebook 2)

Join: `plr_id` (CSV) = `PLR_ID` (Basis)

Neue Spalten:
- `transit_count` — OPNV-Haltestellen im PLR
- `university_count` — Universitäten und Hochschulen
- `school_count` — Schulen
- `nightlife_count` — Bars und Nachtclubs
- `office_count` — Büros (corporate + government)
- `fastfood_count` — Fast-Food-Restaurants (Wettbewerbs-Proxy)

**Hinweis:** Falls `lor_infrastruktur.csv` noch nicht existiert (Notebook 2 noch nicht ausgeführt),
werden alle Infrastruktur-Spalten mit `None` vorbelegt. Notebook 3 kann dann erneut ausgeführt werden.

In [38]:
# ============================================================
# Schritt 7 — Infrastruktur (Google Places Aggregate API)
# Läuft parallel in infrastruktur_aggregate-api.ipynb
# ============================================================

INFRA_COLS = ['transit_count','university_count','school_count',
              'nightlife_count','office_count','fastfood_count']

if not INFRASTRUKTUR_CSV.exists():
    print('WARNUNG: ' + str(INFRASTRUKTUR_CSV) + ' existiert noch nicht.')
    print('Infrastruktur-Spalten werden mit None vorbelegt.')
    print('Notebook erneut ausführen sobald infrastruktur_aggregate-api.ipynb fertig ist.')
    for pid in plr_rows:
        for col in INFRA_COLS:
            plr_rows[pid][col] = None
else:
    df_infra = pd.read_csv(INFRASTRUKTUR_CSV, dtype={'plr_id': str})
    joined = 0
    for _, row in df_infra.iterrows():
        pid = str(row['plr_id']).strip()
        if pid not in plr_rows:
            continue
        for col in INFRA_COLS:
            val = row.get(col)
            plr_rows[pid][col] = int(val) if pd.notna(val) else None
        joined += 1
    print('Infrastruktur gejoint: ' + str(joined) + '/' + str(len(plr_rows)) + ' PLR')


Infrastruktur gejoint: 542/542 PLR


## Schritt 8 — Scores berechnen (0–100)

Auf Basis aller gesammelten Merkmale werden vier normierte Scores (0-100) berechnet.

**Gewichtung des Standort-Gesamtscores:**

| Score-Komponente | Gewicht | Beschreibung |
|-----------------|---------|-------------|
| Nachfrage (Marktlücke) | 30% | Einwohner pro Dönerladen — je höher desto besser |
| ÖPNV-Anbindung | 20% | transit_count — je mehr Haltestellen desto besser |
| Wettbewerb | 20% | (Döner + FastFood) / Einwohner — je geringer desto besser (invertiert) |
| Infrastruktur | 15% | Uni + Büro + Nachtleben — Indikator für Laufkundschaft |
| Altersstruktur | 15% | Anteil 18-35 Jährige — Kernzielgruppe |

Die Normierung erfolgt per Min-Max auf [0, 100]. PLR mit fehlenden Werten für eine Komponente
erhalten einen gewichteten Teilscore über die verfügbaren Komponenten.

In [39]:
# ============================================================
# Schritt 8 — Abgeleitete Kennzahlen
# ============================================================
# KEINE SCORES hier — Scoring erfolgt interaktiv im Dashboard.
# Berechnet werden nur reine Kennzahlen, die direkt aus den
# vorhandenen Spalten ableitbar sind.

for pid, row in plr_rows.items():
    einw  = row.get('einwohner_gesamt') or 0
    doen  = row.get('doener_count') or 0
    ff    = row.get('fastfood_count') or 0
    g_neu = row.get('gastro_neu') or 0
    g_all = row.get('gastro_gesamt') or 0
    mss_s = row.get('mss_status_index')
    mss_d = row.get('mss_dynamik_label') or ''

    # Einwohner pro Doenerladen (Kern-Marktluecken-Indikator)
    row['einwohner_pro_doener'] = round(einw / doen, 1) if doen > 0 else None

    # Wettbewerbs-Index: (Doener + FastFood) / 1000 Einwohner
    row['wettbewerb_index'] = round((doen + ff) / einw * 1000, 3) if einw > 0 else None

    # Gastro-Fluktuation: Anteil neue Betriebe (business_age <= 2)
    row['gastro_fluktuation'] = round(g_neu / g_all, 3) if g_all > 0 else None

    # Gentrification-Flag: niedriger Sozialstatus + positive Dynamik
    if mss_s is not None:
        row['gentrification_flag'] = 1 if (mss_s <= 2 and 'positiv' in mss_d.lower()) else 0
    else:
        row['gentrification_flag'] = None

computed = sum(1 for r in plr_rows.values() if r.get('wettbewerb_index') is not None)
print('Abgeleitete Kennzahlen berechnet fuer ' + str(computed) + '/' + str(len(plr_rows)) + ' PLR')
print('  einwohner_pro_doener, wettbewerb_index, gastro_fluktuation, gentrification_flag')
print('Scores werden NICHT hier berechnet — nur im Dashboard (interaktive Slider).')


Abgeleitete Kennzahlen berechnet fuer 540/542 PLR
  einwohner_pro_doener, wettbewerb_index, gastro_fluktuation, gentrification_flag
Scores werden NICHT hier berechnet — nur im Dashboard (interaktive Slider).


## Schritt 9 — Export: CSV + SQLite

Alle Spalten werden in der definierten Reihenfolge in zwei Ausgabedateien gespeichert:
- `berlin_masterdata.csv` — UTF-8-kodierte CSV für Power BI / Tableau / Python
- `berlin_masterdata.db` — SQLite-Datenbank, Tabelle `planungsraeume`



In [40]:
# ============================================================
# Schritt 9 — Export: CSV + SQLite
# ============================================================
# Alle Rohdaten-Spalten (keine Scores — Scoring erfolgt im Dashboard).

COLUMNS = [
    # Geografie
    'plr_id', 'plr_name', 'bezirk', 'flaeche_km2',
    'centroid_lat', 'centroid_lng',
    # Bevoelkerung
    'einwohner_gesamt', 'einwohner_18_35', 'einwohner_dichte',
    # Einkommen
    'medianeinkommen_eur',
    # Sozialindex
    'mss_status_index', 'mss_dynamik_label',
    # Doenerlaeden
    'doener_count', 'doener_avg_rating', 'doener_total_reviews',
    'doener_best_rating', 'doener_pct_delivery',
    'doener_avg_hours_per_week', 'doener_pct_late_night',
    'doener_pct_open_sunday', 'doener_avg_price_level',
    # Infrastruktur
    'transit_count', 'university_count', 'school_count',
    'nightlife_count', 'office_count', 'fastfood_count',
    # Gewerbedaten
    'gastro_gesamt', 'gastro_neu',
    # Abgeleitete Kennzahlen
    'einwohner_pro_doener', 'wettbewerb_index', 'gastro_fluktuation', 'gentrification_flag',
]

rows = list(plr_rows.values())
df_out = pd.DataFrame(rows, columns=COLUMNS)

# --- CSV Export ---
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print('CSV gespeichert: ' + str(OUTPUT_CSV))
print('Zeilen: ' + str(len(df_out)) + ', Spalten: ' + str(len(df_out.columns)))

# --- SQLite Export ---
import sqlite3
conn = sqlite3.connect(OUTPUT_SQLITE)
df_out.to_sql('planungsraeume', conn, if_exists='replace', index=False)
conn.close()
print('SQLite gespeichert: ' + str(OUTPUT_SQLITE))


CSV gespeichert: C:\Users\cedric\code\VS\Studium\BI_Projekt\.claude\worktrees\tender-knuth\Cedrics_WIP\Data_Pipeline\berlin_masterdata.csv
Zeilen: 542, Spalten: 33
SQLite gespeichert: C:\Users\cedric\code\VS\Studium\BI_Projekt\.claude\worktrees\tender-knuth\Cedrics_WIP\Data_Pipeline\berlin_masterdata.db


## Validierung

Prüft Vollständigkeit (fehlende Werte) und gibt eine numerische Zusammenfassung aus.
Spalten mit mehr als 10% fehlenden Werten werden mit `!!!` markiert.

In [41]:
# ============================================================
# Validierung — Vollständigkeit und Plausibilität
# ============================================================

print('=== Datensatz-Überblick ===')
print('Zeilen (PLR):  ' + str(len(df_out)))
print('Spalten:       ' + str(len(df_out.columns)))
print()
print('--- Fehlende Werte ---')
for col in COLUMNS:
    n = df_out[col].isna().sum()
    pct = n / len(df_out) * 100
    flag = '  !!!' if pct > 10 else ''
    print(f'  {col:<30} {n:>3} fehlend ({pct:.0f}%){flag}')
print()
print('--- Numerische Zusammenfassung ---')
show = ['einwohner_gesamt','doener_count','transit_count',
        'medianeinkommen_eur','einwohner_pro_doener']
print(df_out[show].describe().round(1).to_string())


=== Datensatz-Überblick ===
Zeilen (PLR):  542
Spalten:       33

--- Fehlende Werte ---
  plr_id                           0 fehlend (0%)
  plr_name                         0 fehlend (0%)
  bezirk                           0 fehlend (0%)
  flaeche_km2                      0 fehlend (0%)
  centroid_lat                     0 fehlend (0%)
  centroid_lng                     0 fehlend (0%)
  einwohner_gesamt                 2 fehlend (0%)
  einwohner_18_35                  1 fehlend (0%)
  einwohner_dichte                 2 fehlend (0%)
  medianeinkommen_eur             37 fehlend (7%)
  mss_status_index                 6 fehlend (1%)
  mss_dynamik_label                6 fehlend (1%)
  doener_count                     0 fehlend (0%)
  doener_avg_rating              143 fehlend (26%)  !!!
  doener_total_reviews             0 fehlend (0%)
  doener_best_rating             143 fehlend (26%)  !!!
  doener_pct_delivery            142 fehlend (26%)  !!!
  doener_avg_hours_per_week      144 fehlen